In [1]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"   # or: os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)

In [2]:
import torch
def get_supported_dtype(device: str) -> torch.dtype:
    if device == "cuda" and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    elif device == "cuda" and torch.cuda.is_fp16_supported():
        return torch.float16
    else:
        return torch.float32

In [8]:
# pip install -U "transformers>=4.41" "peft>=0.11" "accelerate>=0.33" torch

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ---- config ----
BASE_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"          # change me
# LORA_ADAPTER_ID = "boreasg/dpo-dpo-v3-test_20260303_162609"     # change me
LORA_ADAPTER_ID = "boreasg/sft-llama8b_test-test_20260206_000112" 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = get_supported_dtype(DEVICE)

# ---- load tokenizer ----
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)

# ---- load two base instances so we can compare the raw base model vs the SFT (PEFT) model ----
device_map = "auto" if DEVICE == "cuda" else None
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=DTYPE,
    device_map=device_map,
)
# load a second copy to apply the adapter onto (keep base_model untouched)
base_for_peft = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=DTYPE,
    device_map=device_map,
)
sft_model = PeftModel.from_pretrained(base_for_peft, LORA_ADAPTER_ID)

# eval mode
base_model.eval()
sft_model.eval()

# quick debug prints
print("DEVICE:", DEVICE)
print("base_model first param dtype:", next(base_model.parameters()).dtype)
print("sft_model first param dtype:", next(sft_model.parameters()).dtype)
print("tokenizer eos_token_id:", tokenizer.eos_token_id)

# ---- generation helper that can run any model instance ----
@torch.inference_mode()
def generate_with_model(model, prompt: str, *, max_new_tokens: int = 512, do_sample: bool = False) -> str:
    messages = [
        {"role": "system", "content": "You are a helpful, concise assistant. Answer directly and accurately."},
        {"role": "user", "content": prompt},
    ]

    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt")
    else:
        text = f"System: You are a helpful, concise assistant. Answer directly and accurately.\nUser: {prompt}\nAssistant:"
        inputs = tokenizer(text, return_tensors="pt")

    # send inputs to the device where the model lives (works with device_map placements)
    model_device = next(model.parameters()).device
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    # sanity-check: single forward to inspect logits
    try:
        logits = model(**inputs).logits
        print(f"Sanity: logits nan? {logits.isnan().any().item()} max_abs={logits.abs().max().item():.3e}")
    except Exception as e:
        print("Sanity forward failed:", e)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )
    output_text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return output_text.strip()

# ---- compare runner ----
def compare_models(prompt: str):
    print("--- Prompt ---")
    print(prompt)
    print()

    print("--- Base model output ---")
    try:
        base_out = generate_with_model(base_model, prompt)
        print(base_out)
    except Exception as e:
        print("Base generation failed:", e)

    print()
    print("--- SFT (PEFT) model output ---")
    try:
        sft_out = generate_with_model(sft_model, prompt)
        print(sft_out)
    except Exception as e:
        print("SFT generation failed:", e)

    return {"base": base_out if 'base_out' in locals() else None, "sft": sft_out if 'sft_out' in locals() else None}

# ---- example ----
if __name__ == "__main__":
    result = compare_models("Explain LoRA in 3 bullet points.")
    # result contains both outputs for programmatic use


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/27.3M [00:00<?, ?B/s]

DEVICE: cuda
base_model first param dtype: torch.bfloat16
sft_model first param dtype: torch.bfloat16
tokenizer eos_token_id: 128009
--- Prompt ---
Explain LoRA in 3 bullet points.

--- Base model output ---
Sanity: logits nan? False max_abs=4.150e+01
LoRA (Learning to be Optimal for RL with Auxiliary Representations) is a technique used in deep learning, particularly in the field of reinforcement learning (RL). Here are 3 key points about LoRA:

• **Weight Tuning**: LoRA is a method for fine-tuning pre-trained models by adding a set of learnable weights on top of the original weights. This allows for efficient adaptation to new tasks or domains without modifying the original weights.

• **Efficient Adaptation**: LoRA achieves efficient adaptation by learning a small set of auxiliary weights that can be added to the original weights. This approach reduces the number of parameters to be learned, making it computationally efficient and memory-friendly.

• **Applicability**: LoRA can be a

In [7]:
compare_models("Write a short essay about RL")

--- Prompt ---
Can you help me write a short humorous piece

--- Base model output ---
Sanity: logits nan? False max_abs=4.150e+01
I'd be happy to help you write a humorous piece. What kind of humor are you aiming for (e.g. sarcasm, puns, witty one-liners, absurdity)? And what's the topic or theme you'd like to write about?

--- SFT (PEFT) model output ---
Sanity: logits nan? False max_abs=4.275e+01
What kind of humorous piece are you thinking of writing (e.g. short story, joke, poem, etc.) and what's the topic or theme?


{'base': "I'd be happy to help you write a humorous piece. What kind of humor are you aiming for (e.g. sarcasm, puns, witty one-liners, absurdity)? And what's the topic or theme you'd like to write about?",
 'sft': "What kind of humorous piece are you thinking of writing (e.g. short story, joke, poem, etc.) and what's the topic or theme?"}

In [10]:
compare_models("Can you give me some restaurant recommendations for tonight in NY")

--- Prompt ---
Can you give me some restaurant recommendations for tonight in NY

--- Base model output ---
Sanity: logits nan? False max_abs=4.150e+01
New York City has a diverse dining scene. Here are some popular restaurant recommendations for different cuisines:

**Fine Dining:**

1.  Eleven Madison Park (3 Michelin stars) - upscale American cuisine in the Flatiron District.
2. Le Bernardin (3 Michelin stars) - seafood-focused French cuisine in Midtown.
3. Per Se (3 Michelin stars) - upscale American cuisine with a focus on seasonal ingredients in the Upper East Side.

**Italian:**

1. Carbone (Midtown) - retro Italian-American cuisine with a lively atmosphere.
2. L'Artusi (Chelsea) - upscale Italian cuisine with a focus on seasonal ingredients.
3. Scarpetta (Multiple locations) - modern Italian cuisine with a focus on seasonal ingredients.

**Asian:**

1. Peter Luger Steak House (Williamsburg, Brooklyn) - high-end steakhouse with a focus on Japanese cuisine.
2. Momofuku Noodle Bar

{'base': "New York City has a diverse dining scene. Here are some popular restaurant recommendations for different cuisines:\n\n**Fine Dining:**\n\n1.  Eleven Madison Park (3 Michelin stars) - upscale American cuisine in the Flatiron District.\n2. Le Bernardin (3 Michelin stars) - seafood-focused French cuisine in Midtown.\n3. Per Se (3 Michelin stars) - upscale American cuisine with a focus on seasonal ingredients in the Upper East Side.\n\n**Italian:**\n\n1. Carbone (Midtown) - retro Italian-American cuisine with a lively atmosphere.\n2. L'Artusi (Chelsea) - upscale Italian cuisine with a focus on seasonal ingredients.\n3. Scarpetta (Multiple locations) - modern Italian cuisine with a focus on seasonal ingredients.\n\n**Asian:**\n\n1. Peter Luger Steak House (Williamsburg, Brooklyn) - high-end steakhouse with a focus on Japanese cuisine.\n2. Momofuku Noodle Bar (East Village) - trendy Asian-fusion cuisine with a focus on noodles.\n3. Sushi Saito (Multiple locations) - high-end sushi 